# Displacement

**Project:** Pali Wacth

**Notebook:** `displacement.ipynb`

**Author:** Yuseof J

**Created:** 18/11/2026 | **Last updated:** 22/11/2026

## Purpose
   This file analyzes displacement pressures by governorate

## Inputs

- `data/raw/boundaries/`: HDX admin boundaries (COD-AB)

## Outputs
- `data/processed/[filename].csv`: what it contains

## Notes and caveats
    - Data runs from Jan 2025 to Oct 2026, so historical analysis is limited accordingly

In [ ]:
import datetime
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns
from functools import reduce

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

CRS = 'EPSG:4326'

# Data imports / exports

In [ ]:
# import paths
PATH_TO_DISPLACEMENT_EVENTS = '../data/raw/pse_idmc_idu_events.csv'
PATH_TO_ADMIN_BOUNDARIES = '../data/raw/pse_admin_boundaries.shp/pse_admin2.shp'

# export paths
PATH_TO_GEOLOCATED_DISPLACEMENT_SUMMARY_GPKG = '../data/processed/<filename.gpkg>'
PATH_TO_DISPLACEMENT_SUMMARY_CSV = '../data/processed/displacement/displacement_summary.csv'

# import data
df_displacement = pd.read_csv(PATH_TO_DISPLACEMENT_EVENTS)
gdf_boundaries = gpd.read_file(PATH_TO_ADMIN_BOUNDARIES)

# Preprocessing

In [ ]:
date_col_to_use = 'displacement_end_date'

# format date columns
df_displacement[date_col_to_use] = pd.to_datetime(df_displacement[date_col_to_use])

# create date grouping columns
df_displacement['YEAR'] = df_displacement[date_col_to_use].dt.year
df_displacement['YEARMO'] = df_displacement[date_col_to_use].dt.to_period('M').astype(str)
df_displacement['YEAR_WEEK'] = df_displacement[date_col_to_use].dt.to_period('W').astype(str)

In [ ]:
# check for duplicate events (will inflate displacement figures)
len_before = len(df_displacement)
df_displacement = df_displacement.drop_duplicates(subset=['latitude', 'longitude', 'figure', 'displacement_date'])
print(f"Dropped {len_before - len(df_displacement)} diplicate rows")

In [ ]:
# rows with missing figures
assert(df_displacement.figure.isna().any() == False)

# EDA

In [ ]:
# df_displacement.columns.tolist()

In [ ]:
# df_displacement.head()

## Geographic Distribution

In [ ]:
print(f"{df_displacement.centroid.nunique()} unique centroids")
print(f"{df_displacement.latitude.nunique()} unique latitudes")
print(f"{df_displacement.longitude.nunique()} unique longitudes")
print(f"{df_displacement.locations_type.nunique()} unique locations_type")
print(f"{df_displacement.locations_name.nunique()} unique locations_name")

### TODO: rich info in locations_name, can be distilled for finer geographic granularity

In [ ]:
# df_displacement.locations_name.tolist()

## Temporal Distribution

### TODO: date columns don't always align with 'description' column (which contains dates of displacement)

In [ ]:
print(f"Earliest start date: {df_displacement.displacement_start_date.min()}")
print(f"Latest start date: {df_displacement.displacement_start_date.max()}")
print(f"Earliest end date: {df_displacement.displacement_end_date.min()}")
print(f"Latest end date: {df_displacement.displacement_end_date.max()}")

# Geoprocessing

In [ ]:
# convert event coordinates into spatial points
gdf_displacement = gpd.GeoDataFrame(
    df_displacement,
    geometry=gpd.points_from_xy(
        df_displacement['longitude'],
        df_displacement['latitude']
    ),
    crs=CRS
)

In [ ]:
# check and align CRSs
print("Events CRS:", gdf_displacement.crs)
print("Admin CRS:", gdf_boundaries.crs)
gdf_displacement = gdf_displacement.to_crs(gdf_boundaries.crs) # failsafe -- convert events CRS to admin boundaries CRS if not already aligned
assert(gdf_displacement.crs == gdf_boundaries.crs)
print('Confirmed CRSs are aligned')

In [ ]:
# perform spatial join (assign event coordinates to an administrative boundary)
gdf_displacement_joined = gpd.sjoin(
    gdf_displacement,
    gdf_boundaries[['adm1_name', 'adm1_pcode', 'adm2_name', 'adm2_pcode', 'geometry']],
    how='left',
    predicate='within'
)

In [ ]:
# check for any duplicate rows after join
assert(len(gdf_displacement_joined) == len(gdf_displacement))

In [ ]:
print("Total events:", len(gdf_displacement_joined))
print("Matched events:", gdf_displacement_joined['adm1_name'].notna().sum())
print("Unmatched events:", gdf_displacement_joined['adm1_name'].isna().sum())

### TODO: handle unmatched events

In [ ]:
unmatched = gdf_displacement_joined[gdf_displacement_joined['adm1_name'].isna()]
if len(unmatched) > 0:
    display(unmatched.head())
else:
    print('All events matched successfully!')

# Plot: Displacement by Gov Over Time

In [ ]:
def plot_displacement_trends(
    df,
    value_col,
    date_col,
    separate=False,
    cumulative=False,
    show_values=True
):

    plot_df = df.copy()

    # Format 'YEAR_WEEK' so that it works with the funtctions below 
    # e.g. the week range "2026-09-21/2026-09-27" has to be a single date
    # this solution will take the end date of the range    if date_granularity == 'YEAR_WEEK':
    plot_df[date_col] = pd.to_datetime(
        plot_df[date_col].str.split('/').str[1]
    )

    plot_df[date_col] = pd.to_datetime(plot_df[date_col])

    # Sort chronologically
    plot_df = plot_df.sort_values(date_col)

    # Calculate cumulative values independently per geography
    plot_col = value_col

    if cumulative:
        plot_col = f'{value_col}_cumulative'
        plot_df[plot_col] = (
            plot_df.groupby('adm2_name')[value_col].cumsum()
        )

    if separate:
        geographies = plot_df['adm2_name'].unique()

        fig, axes = plt.subplots(
            len(geographies), 1,
            figsize=(14, 5 * len(geographies)),
            sharex=True
        )

        if len(geographies) == 1:
            axes = [axes]

        for geo, ax in zip(geographies, axes):
            subset = plot_df[plot_df['adm2_name'] == geo]

            sns.lineplot(
                data=subset,
                x=date_col,
                y=plot_col,
                marker='o',
                ax=ax
            )

            # Display values at each point
            if show_values:
                for _, row in subset.iterrows():
                    ax.annotate(
                        f'{row[plot_col]:,.0f}',
                        (row[date_col], row[plot_col]),
                        textcoords='offset points',
                        xytext=(0, 8),
                        ha='center',
                        fontsize=8
                    )

            ax.set_title(geo)
            ax.set_ylabel(plot_col)

        plt.xlabel('Date')

    else:
        fig, ax = plt.subplots(figsize=(14, 6))

        sns.lineplot(
            data=plot_df,
            x=date_col,
            y=plot_col,
            hue='adm2_name',
            marker='o',
            ax=ax
        )

        # Display values at each point
        if show_values:
            for _, row in plot_df.iterrows():
                ax.annotate(
                    f'{row[plot_col]:,.0f}',
                    (row[date_col], row[plot_col]),
                    textcoords='offset points',
                    xytext=(0, 8),
                    ha='center',
                    fontsize=8
                )

        ax.set_title(f'{plot_col} Over Time by Geography')
        ax.set_xlabel('Date')
        ax.set_ylabel(plot_col)
        ax.legend(title='Geography')

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
#date_granularity = 'YEARMO' 
date_granularity = 'YEAR_WEEK'
regression_group = ['adm2_name', date_granularity]

# get data in proper format: region, yearmo-week
gdf_displacement_rate = gdf_displacement_joined.groupby(regression_group).agg(
    num_events=('figure', 'size'),
    num_displaced=('figure', 'sum')
)

In [ ]:
# filter out gaza (displacement figures dwarf WB values)
wb_admins = gdf_displacement_joined[gdf_displacement_joined.adm1_name=='West Bank'].adm2_name.unique()
df = gdf_displacement_rate.reset_index().copy()
wb_filter = df.adm2_name.isin(wb_admins)
#df_wb = df[wb_filter]

In [ ]:
#df_wb

In [ ]:
# plot_displacement_trends(df_wb, 'num_displaced', date_granularity, separate=True)

NOTE: treating the displacement end date as the date that everyone from this row was displaced, even though they were likely displaced over the period from start to finish date

In [ ]:
#df_displacement[['event_end_date', 'YEAR', 'YEARMO', 'YEAR_WEEK']]

### VIZ: Cumulative displacement by governorate

Types of responses:

    A. Sustained presence, mobilize resources (informed by projected num_displaced) (i.e. have shelter situations figured out for enough people)
    B. Track historically displaced people, ensure they are in stable situations
    C. Surge response: mobilize resources qucikly to prepare for sudden demand
    D. Refocus presence alsewhere, but keep tracking situation in case problem re-intensifies
    E. Popup presence, mobilize resources as needed, consider keeping operations 

Types of displacement trends observed:

1. Sustained increase throughout (ex Nablus) 
        - suggested response: A, B
2. Sustained increase, stop (ex Bethlehem)  
        - suggested response: B
3. Sustained increase, lull, increase (ex Jerusalem)

Four State System (see paper notes) 

In [ ]:
plot_displacement_trends(df[wb_filter], 'num_displaced', date_granularity, separate=True, cumulative=True)

# Q1: Where is displacement happening?

## TODO: Normalize event counts by population since comparing regions of different sizes.

## Setup

In [ ]:
displacement_date_col = 'displacement_end_date'

# date cutoffs
latest_date = gdf_displacement_joined[displacement_date_col].max()
one_year_ago = latest_date - pd.DateOffset(years=1)
three_months_ago = latest_date - pd.DateOffset(months=3)
one_month_ago = latest_date - pd.DateOffset(months=1)

# create filtered values
gdf_displacement_joined['last_year'] = (
    gdf_displacement_joined['figure']
    .where(gdf_displacement_joined[displacement_date_col] >= one_year_ago, 0)
)
gdf_displacement_joined['last_3_months'] = (
    gdf_displacement_joined['figure']
    .where(gdf_displacement_joined[displacement_date_col] >= three_months_ago, 0)
)
gdf_displacement_joined['last_month'] = (
    gdf_displacement_joined['figure']
    .where(gdf_displacement_joined[displacement_date_col] >= one_month_ago, 0)
)
gdf_displacement_joined['previous_9_months'] = (
    gdf_displacement_joined['figure']
    .where(
        gdf_displacement_joined[displacement_date_col].between(
            one_year_ago, three_months_ago, inclusive='left'
        ), 0
    )
)

## Total num displacement events per admin 

In [ ]:
# Aggregate event counts by ADM1 and ADM2
result = (
    gdf_displacement_joined
    .assign(
        last_year=lambda x: x[displacement_date_col] >= one_year_ago,
        previous_9_months=lambda x: x[displacement_date_col].between(one_year_ago, three_months_ago, inclusive='left'),
        last_3_months=lambda x: x[displacement_date_col] >= three_months_ago,
        last_1_month=lambda x: x[displacement_date_col] >= one_month_ago,
    )
    .groupby(['adm1_name'])
    .agg(
        total_events=('adm2_name', 'size'),
        last_year=('last_year', 'sum'),
        previous_9_months=('previous_9_months', 'sum'),
        last_3_months=('last_3_months', 'sum'),
        last_1_month=('last_1_month', 'sum'),
    )
    .sort_values(
        ['adm1_name', 'total_events'],
        ascending=[True, False]
    )
)

result

In [ ]:
# Aggregate event counts by ADM1 and ADM2
df_events_summary = (
    gdf_displacement_joined
    .assign(
        last_year=lambda x: x[displacement_date_col] >= one_year_ago,
        previous_9_months=lambda x: x[displacement_date_col].between(one_year_ago, three_months_ago, inclusive='left'),        
        last_3_months=lambda x: x[displacement_date_col] >= three_months_ago,
        last_1_month=lambda x: x[displacement_date_col] >= one_month_ago,
    )
    .groupby(['adm1_name', 'adm2_name'])
    .agg(
        total=('adm2_name', 'size'),
        last_year=('last_year', 'sum'),
        previous_9_months=('previous_9_months', 'sum'),
        last_3_months=('last_3_months', 'sum'),
        last_1_month=('last_1_month', 'sum'),
    )
    .sort_values(
        ['adm1_name', 'total'],
        ascending=[True, False]
    )
)

# column renaming happens after creating df_displaced_per_event
#df_events_summary = df_events_summary.add_prefix("num_events_")

df_events_summary

## Total num displaced people per admin (non-cumaltive)

In [ ]:
# aggregate
result = (
    gdf_displacement_joined
    .groupby(['adm1_name'])
    .agg(
        total=('figure', 'sum'),
        last_year=('last_year', 'sum'),
        last_3_months=('last_3_months', 'sum'),
        last_1_month=('last_month', 'sum')
    )
    .sort_values(
        ['adm1_name', 'total'],
        ascending=[True, False]
    )
)

result

In [ ]:
# aggregate
df_displaced_summary = (
    gdf_displacement_joined
    .groupby(['adm1_name', 'adm2_name'])
    .agg(
        total=('figure', 'sum'),
        last_year=('last_year', 'sum'),
        previous_9_months=('previous_9_months', 'sum'),
        last_3_months=('last_3_months', 'sum'),
        last_1_month=('last_month', 'sum')
    )
    .sort_values(
        ['adm1_name', 'total'],
        ascending=[True, False]
    )
)

# column renaming happens after creating df_displaced_per_event
#df_displaced_summary = df_displaced_summary.add_prefix("num_displaced_")

df_displaced_summary

## Days since last displacement (as of max date in dataset, not today)

In [ ]:
# Calculate days since last displacement in dataset
reference_date = gdf_displacement_joined[displacement_date_col].max()
print(f"\nAs of {reference_date}: \n")

# Get most recent displacement date per ADM2
latest_dates = (
    gdf_displacement_joined
    .groupby('adm2_name')[displacement_date_col]
    .max()
    .rename('last_displacement')
)

# Join latest dates back to original data
gdf_displacement_recency = (
    gdf_displacement_joined
    .merge(latest_dates, on='adm2_name')
)

# Keep only events occurring on the most recent date
df_displacement_recency = (
    gdf_displacement_recency[
        gdf_displacement_recency[displacement_date_col]
        == gdf_displacement_recency['last_displacement']
    ]
    .groupby(['adm1_name', 'adm2_name', 'last_displacement'])
    .agg(people_displaced_last_day=('figure', 'sum'))
    .reset_index()
)

df_displacement_recency['days_since_last_displacement'] = (
    reference_date - df_displacement_recency['last_displacement']
).dt.days

# Sort by recency
df_displacement_recency = df_displacement_recency.sort_values(
    'days_since_last_displacement'
)

df_displacement_recency

# Q2: Where is displacement changing the fastest?

In [ ]:
# gdf_displacement.head()

## Deterioration Rate

Here we are looking at newly displaced, not cumulative 

The idea here is to answer the "low/high acceleration" part of the 4 state definitions (e.g. (1) low displacement + low acceleration) 

"Acceleration" here is defined by comparing the most recent 3 months to the 9 months that preceded it (avg_rate_3_month / avg_rate_9)_month

In [ ]:
# aggregate
df_deterioration = (
    gdf_displacement_joined
    .groupby(['adm1_name', 'adm2_name'])
    .agg(
        last_year=('last_year', 'sum'),
        previous_9_months=('previous_9_months', 'sum'),
        last_3_months=('last_3_months', 'sum'),
        last_1_month=('last_month', 'sum')
    )
    .sort_values(
        ['adm1_name', 'last_3_months'],
        ascending=[True, False]
    )
)

# divide each row by its corresponding number of months
divisors = {
    'last_year': 12,
    'previous_9_months': 9,
    'last_3_months': 3,
    'last_1_month': 1
}

df_deterioration[list(divisors)] = df_deterioration[list(divisors)].div(
    pd.Series(divisors)
)

# e.g. 2x more people displaced per month in last 3 months vs previous 9 months
df_deterioration['deterioration_ratio'] = df_deterioration['last_3_months'] / df_deterioration['previous_9_months']
# result['deterioration_ratio'] = (result['last_3_months'] - result['previous_9_months']) / result['previous_9_months']

skip = ["deterioration_ratio"]
df_deterioration = df_deterioration.rename(columns=lambda col: col if col in skip else f"displaced_per_month_{col}")

print("\n------------------------------")
print("   NUM DISPLACED PER MONTH")
print("------------------------------\n")

# format numbers for printout
df_deterioration.select_dtypes(include="number").astype(float).apply(lambda s: s.map(lambda x: float(f"{x:.2g}")))

## TODO: Surge Detection

Surges are an important part of understanding deterioiration. They tell us:

    - alot more people were displaced than anticipated (meaning resources were likely not mobilized quickly enough to handle it)
   
    - need for org presence may have shifted (e.g. surge + ongoing means an org needs to be established where previously there may not have been a need)

Surge label (ongoing, level shift, or spike and return), with surge share and months since surge. It tells the reader whether the deterioration is lasting, which the ratio alone can’t.

## Frequency vs Severity

This explains WHY a governorate is worsening, which matters for choosing a response. Is the deterioration being driven by more events, more severe events, or a combination of both


Deterioration Driver:

                              (frequency)           (severity)
    Displacement_per_month = events_per_month x displaced_per_event

### Events Per Month (Frequency)

In [ ]:
# Aggregate event counts by ADM1 and ADM2
df_events_per_month = (
    gdf_displacement_joined
    .assign(
        last_year=lambda x: x[displacement_date_col] >= one_year_ago,
        previous_9_months=lambda x: x[displacement_date_col].between(one_year_ago, three_months_ago, inclusive='left'),
        last_3_months=lambda x: x[displacement_date_col] >= three_months_ago,
        last_1_month=lambda x: x[displacement_date_col] >= one_month_ago,
    )
    .groupby(['adm1_name', 'adm2_name'])
    .agg(
        # total_events=('adm2_name', 'size'),
        last_year=('last_year', 'sum'),
        previous_9_months=('previous_9_months', 'sum'),
        last_3_months=('last_3_months', 'sum'),
        last_1_month=('last_1_month', 'sum'),
    )
    .sort_values(
        ['adm1_name', 'last_year'],
        ascending=[True, False]
    )
)

# divide each row by its corresponding number of months
divisors = {
    'last_year': 12,
    'previous_9_months': 9,
    'last_3_months': 3,
    'last_1_month': 1
}

df_events_per_month[list(divisors)] = df_events_per_month[list(divisors)].div(
    pd.Series(divisors)
)

# how many more events in most recent 3 months compared to previous 9 months? (e.g. 2x more events per month)
df_events_per_month['event_frequency_ratio'] = df_events_per_month['last_3_months'] / df_events_per_month['previous_9_months']

skip = ["event_frequency_ratio"]
df_events_per_month = df_events_per_month.rename(columns=lambda col: col if col in skip else f"events_per_month_{col}")

print("\n------------------------------")
print("   NUM EVENTS PER MONTH")
print("------------------------------\n")

# format numbers for printout
df_events_per_month.select_dtypes(include="number").astype(float).apply(lambda s: s.map(lambda x: float(f"{x:.3g}")))

### Displaced per Event (Severity)

In [ ]:
df_displaced_per_event = df_displaced_summary / df_events_summary

# how many more displaced per event in most recent 3 months compared to previous 9 months? (e.g. 2x more displaced per event)
df_displaced_per_event['severity_ratio'] = df_displaced_per_event['last_3_months'] / df_displaced_per_event['previous_9_months']

skip = ["severity_ratio"]
df_displaced_per_event = df_displaced_per_event.rename(columns=lambda col: col if col in skip else f"displaced_per_event_{col}")

print("\n------------------------------")
print("   Num Displaced Per Event")
print("------------------------------\n")

# format numbers for printout
df_displaced_per_event.select_dtypes(include="number").astype(float).apply(lambda s: s.map(lambda x: float(f"{x:.2g}")))

# Export Metrics

Rename columns before joining

In [ ]:
df_events_summary = df_events_summary.add_prefix("num_events_")
df_displaced_summary = df_displaced_summary.add_prefix("num_displaced_")

 Combine the metrics above are combined into one dataframe to create a cohesive summary of the state of displacement in each governorate

In [ ]:
# round up all the dfs
displacement_dfs = [df_events_summary,
                    df_displaced_summary,
                    df_displacement_recency,
                    df_deterioration,
                    df_events_per_month,
                    df_displaced_per_event
                   ]

# reset index on all df's if multiindex (so they can be joined properly)
displacement_dfs_reset = [df.reset_index() if isinstance(df.index, pd.MultiIndex) else df for df in displacement_dfs]

# join all the dfs on adm1_name, adm2_name to get all metrics per governorate
df_displacement_combined = reduce(
    lambda left, right: pd.merge(
        left, right,
        on=['adm1_name', 'adm2_name'],
        how='outer'
    ),
    displacement_dfs_reset
)

In [ ]:
df_displacement_combined.to_csv(PATH_TO_DISPLACEMENT_SUMMARY_CSV, index=False)

# TODO: Export as a gdf for maps